In [ ]:
# Import libraries
import os, random, pathlib, math, json, time
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid, save_image
from torchvision.transforms import functional as F
from tqdm import tqdm
from torchvision.utils import make_grid
from IPython.display import display
import matplotlib.pyplot as plt
%matplotlib inline
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", DEVICE)

In [ ]:
# Import data
from src.data_processing import load_pt_bundle
# Import N2S dataset 
from src.dataset import GlomeruliN2SDataset, generate_mask

# Import U-Net model
from src.model import GlomeruliUNet

# Import N2S loss and metrics
from src.baseline_models import n2s_loss, masked_psnr

# Import debiased PSNR functions
from src.baseline_models import estimate_sigma_batch, debiased_psnr



In [ ]:
#load data
train_pt = 'path to train data'

test_pt = 'path to test data'

train_imgs, train_labels = load_pt_bundle(train_pt)
test_imgs,  test_labels  = load_pt_bundle(test_pt)

batch_size = 32
train_ds = GlomeruliN2SDataset(train_imgs, train_labels, p_drop=0.10)
test_ds  = GlomeruliN2SDataset(test_imgs,  test_labels,  p_drop=0.10)  

train_loader = DataLoader(train_ds, batch_size, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train images: {len(train_ds)}  |  Test images: {len(test_ds)}")


In [ ]:
model = GlomeruliUNet().to(DEVICE)
print(sum(p.numel() for p in model.parameters())/1e6, "M parameters")

In [ ]:
print("train batches :", len(train_loader))
print("test batches :", len(test_loader))

In [ ]:
from pathlib import Path
ckpt_dir = Path("path to checkpoints directory")
ckpt_dir.mkdir(parents=True, exist_ok=True)
epochs = 50
lr = 1e-5
optimizer = torch.optim.Adam(model.parameters(), lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, factor=0.5, patience=5
)
log_every = max(1, len(train_loader)//5) # five prints per epoch
plot_every = 1 # epochs
for epoch in range(1, epochs + 1):
    # ---------- training ----------
    model.train()
    running = 0.0
    for i, (masked, target, mask) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}"), 1):
        masked, target, mask = [t.to(DEVICE, non_blocking=True) for t in (masked, target, mask)]
        optimizer.zero_grad()
        pred = model(masked)
        loss = n2s_loss(pred, target, mask)
        loss.backward()
        optimizer.step()
        running += loss.item()
        if i % log_every == 0 or i == len(train_loader):
            avg = running / log_every
            print(f" batch {i:4d}/{len(train_loader)} loss {avg:.4f}", flush=True)
            running = 0.0
    # ---------- validation ----------
    model.eval()
    val_loss = 0.; val_psnr = 0.; n_val = 0
    with torch.no_grad():
        for masked, noisy, mask in test_loader:
            masked, noisy, mask = [t.to(DEVICE, non_blocking=True) for t in (masked, noisy, mask)]
            pred = model(masked)
            val_loss += n2s_loss(pred, noisy, mask).item()
            sigma_hat = estimate_sigma_batch(noisy.cpu()).to(noisy.device)
            val_psnr += debiased_psnr(pred, noisy, mask, sigma_hat)
            n_val += 1
    val_loss /= n_val
    val_psnr /= n_val
    scheduler.step(val_loss)
    print(f"=> Epoch {epoch:02d} | val‑loss {val_loss:.4f} | est‑PSNR {val_psnr:.2f} dB | lr {scheduler.get_last_lr()[0]:.1e}")
    # ---------- quick visual check ----------
    if epoch % plot_every == 0:
        masked, target, mask = next(iter(test_loader))
        with torch.no_grad():
            denoised = model(masked.to(DEVICE)).cpu()
        from torchvision.utils import make_grid
        import matplotlib.pyplot as plt
        grid = make_grid(torch.cat([masked[:6], denoised[:6], target[:6]], 0),
                        nrow=6, normalize=True, value_range=(0,1), padding=2)
        plt.figure(figsize=(15, 5))
        plt.axis('off')
        plt.title(f'Epoch {epoch}: masked / denoised / noisy rows')
        plt.imshow(grid.permute(1, 2, 0).numpy())
        plt.show()
    # ---------- checkpoint ----------
    torch.save({"epoch": epoch,
               "model_state": model.state_dict(),
               "opt_state": optimizer.state_dict()},
               ckpt_dir / f"n2s_epoch{epoch:02d}.pt")

In [ ]:
# Save denoised images
import os
from torchvision.utils import save_image

# Create output directory
output_dir = Path("path to n2s_denoised_images")
output_dir.mkdir(parents=True, exist_ok=True)

print("Generating and saving denoised images...")
model.eval()
with torch.no_grad():
    for i, (masked, target, mask) in enumerate(tqdm(test_loader, desc="Saving denoised images")):
        masked = masked.to(DEVICE)
        denoised = model(masked).cpu()
        
        # Save batch of denoised images
        for j in range(denoised.shape[0]):
            img_idx = i * batch_size + j
            save_image(denoised[j], output_dir / f"denoised_{img_idx:04d}.png", normalize=True)

print(f"Denoised images saved to: {output_dir}")
print(f"Total images saved: {len(test_ds)}")